In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("EnergyForecasting") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Define schema for smart meter CSV data
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("Power_Consumption", DoubleType(), True),
    StructField("voltage", DoubleType(), True),
    StructField("current", DoubleType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True)
])

# Option 1: Read single CSV file
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .csv("/content/smart_grid_dataset.csv")


In [2]:
# Structured Streaming with CSV files
# Useful for processing new CSV files as they arrive

def setup_csv_streaming(input_path, checkpoint_path):
    """
    Monitor directory for new CSV files and process them
    """
    streaming_df = spark \
        .readStream \
        .option("header", "true") \
        .schema(schema) \
        .option("maxFilesPerTrigger", 1) \
        .csv(input_path)

    # Process streaming data
    processed_df = streaming_df \
        .withColumn("processing_time", current_timestamp())

    return processed_df



## Energy Consumption Forecasting Pipeline

This notebook implements a comprehensive energy consumption forecasting pipeline using PySpark and TensorFlow/Keras. The pipeline includes data loading, preprocessing, feature engineering, training of a hybrid forecasting model (LSTM and ensemble models), and visualization of results. The final predictions can also be saved and exported to Google Drive.

### 1. Spark Session Initialization and Data Loading

This section initializes a SparkSession for distributed data processing and defines the schema for the smart meter data. It then loads a single CSV file into a Spark DataFrame for initial processing.

### 2. Structured Streaming Setup (Optional)

This function provides a template for setting up a structured streaming pipeline. It allows the system to monitor a directory for new CSV files and process them as they arrive, enabling real-time or near real-time forecasting. This part is not actively used in the batch processing pipeline but is available for streaming scenarios.

### 3. Data Preprocessing

The `preprocess_csv_data` function cleans and prepares the raw smart meter data. This involves:
*   **Handling Missing Values**: Filling numerical columns with appropriate values (e.g., 0 for `Power_Consumption`, mean for `temperature` and `humidity`).
*   **Removing Duplicates**: Eliminating duplicate records based on the `timestamp`.
*   **Sorting Data**: Ensuring data is sorted chronologically by `timestamp`, which is crucial for time series analysis.
*   **Outlier Filtering**: Applying the Interquartile Range (IQR) method to filter out extreme outliers in `Power_Consumption`.

### 4. Feature Engineering

The `create_features` function generates a rich set of features from the preprocessed data, which are essential for improving the accuracy of forecasting models. These features include:
*   **Temporal Features**: Extracting hour, day of week, day of month, month, quarter, year, and week of year from the `timestamp`.
*   **Binary Features**: Creating indicators for weekends, business hours, and peak hours.
*   **Lag Features**: Incorporating past `Power_Consumption` values (e.g., 1 hour, 24 hours, 1 week ago) to capture temporal dependencies.
*   **Rolling Window Features**: Calculating moving averages, standard deviations, minimums, and maximums of `Power_Consumption` over defined windows (e.g., 24 hours, 7 days).
*   **Weather Interaction Features**: Creating interaction terms between weather variables (temperature, humidity) and temporal features to capture combined effects.
*   **Renaming Target Variable**: Renaming `Power_Consumption` to `energy_kwh` for consistency.

### 5. Train-Test Split

The `create_train_test_split` function divides the dataset into training and testing sets based on a chronological split. This approach is vital for time series forecasting to ensure that the model is evaluated on future, unseen data, mimicking real-world prediction scenarios.

### 6. LSTM Model Training

This section handles the preparation of data for the LSTM model and its training.
*   **`prepare_lstm_data`**: Converts a Spark DataFrame to NumPy arrays suitable for LSTM, applies `StandardScaler` for normalization, and creates sequences for time series prediction.
*   **`build_advanced_lstm`**: Constructs a Bidirectional LSTM model using TensorFlow/Keras, featuring multiple LSTM layers, Dropout for regularization, and Dense layers for output. The model is compiled with the Huber loss function and Adam optimizer.
*   The code then prepares training and testing data, builds the LSTM model, and trains it using `model.fit` with early stopping and learning rate reduction callbacks.

### 7. Ensemble Model Training (Random Forest & Gradient Boosted Trees)

The `train_ensemble_models` function trains two traditional machine learning models from Apache Spark MLlib: Random Forest Regressor and Gradient Boosted Trees Regressor.
*   It defines a set of relevant features and uses `VectorAssembler` to combine them into a single feature vector.
*   Both models are trained on the prepared training data.
*   The performance of each model is evaluated on the test set using RMSE, MAE, and R² metrics.
*   The trained models are saved for later use in the hybrid predictor.

### 8. Hybrid Energy Predictor Class

The `HybridEnergyPredictor` class integrates the trained LSTM and ensemble models to provide a combined, more robust prediction.
*   It loads the pre-trained LSTM model, the `StandardScaler` used for LSTM data, and the Random Forest and GBT models.
*   `predict_lstm` and `predict_ensemble` methods handle individual model predictions.
*   The `hybrid_predict` method combines the predictions from LSTM, Random Forest, and GBT models using a weighted average, aligning array sizes to account for sequence length differences in LSTM.

### 9. Batch Prediction and Saving Results

The `batch_predict_and_save` function provides a utility for making predictions on new, incoming CSV data.
*   It reads new data, preprocesses it, and applies feature engineering, just like the training pipeline.
*   It then uses the `HybridEnergyPredictor` to generate forecasts.
*   Predictions are added to a Pandas DataFrame, along with a simple prediction interval.
*   The results are saved to a CSV file and also as a Parquet file for efficient storage and retrieval.

### 10. Main Forecasting Pipeline

The `main_pipeline` function orchestrates the entire energy forecasting process from start to finish.
*   It loads data, preprocesses it, and performs feature engineering.
*   Splits data into training and testing sets chronologically.
*   Trains the LSTM model and the ensemble models (Random Forest and GBT).
*   Initializes the `HybridEnergyPredictor` and makes predictions on the test set.
*   Evaluates the hybrid model's performance using RMSE, MAE, R², and MAPE.
*   Saves the predictions to a CSV file.

This function provides a complete workflow for training and evaluating the forecasting system.

### 11. Visualization of Results

The `visualize_results` function generates plots to visually assess the model's performance:
*   **Time Series Plot**: Compares actual energy consumption with predicted values over time.
*   **Scatter Plot**: Displays actual vs. predicted values to show the correlation and spread of predictions.

The plots are saved as PNG files in a specified directory.

### 12. Google Drive Integration

This section sets up Google Drive integration to persistent model artifacts, predictions, and plots. This ensures that all generated outputs are saved to a user's Google Drive, preventing data loss and making it easy to access the results outside of the Colab environment.

*   Mounts Google Drive to the Colab environment.
*   Creates a dedicated output directory and subdirectories (`models`, `predictions`, `plots`, `processed_data`) within Google Drive.
*   Copies all relevant output files (models, scalers, prediction CSVs/Parquets, plots) from the local Colab environment to the designated Google Drive folders.

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

def preprocess_csv_data(df):
    """
    Clean and prepare CSV data for modeling
    """
    # Handle missing values
    df = df.na.fill({
        'Power_Consumption': 0.0,
        'temperature': df.agg({'temperature': 'mean'}).collect()[0][0],
        'humidity': df.agg({'humidity': 'mean'}).collect()[0][0]
    })

    # Remove duplicates
    df = df.dropDuplicates(['timestamp'])

    # Sort by timestamp
    df = df.orderBy('timestamp')

    # Filter outliers (using IQR method)
    quantiles = df.approxQuantile('Power_Consumption', [0.25, 0.75], 0.05)
    Q1, Q3 = quantiles[0], quantiles[1]
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df = df.filter(
        (col('Power_Consumption') >= lower_bound) &
        (col('Power_Consumption') <= upper_bound)
    )

    return df

# Apply preprocessing
clean_df = preprocess_csv_data(df)

In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

def create_features(df):
    """
    Create comprehensive feature set for energy forecasting
    """
    # Temporal features
    df = df \
        .withColumn("hour", hour("timestamp")) \
        .withColumn("day_of_week", dayofweek("timestamp")) \
        .withColumn("day_of_month", dayofmonth("timestamp")) \
        .withColumn("month", month("timestamp")) \
        .withColumn("quarter", quarter("timestamp")) \
        .withColumn("year", year("timestamp")) \
        .withColumn("week_of_year", weekofyear("timestamp"))

    # Binary features
    df = df \
        .withColumn("is_weekend",
                   when(col("day_of_week").isin([1, 7]), 1).otherwise(0)) \
        .withColumn("is_business_hours",
                   when((col("hour") >= 9) & (col("hour") <= 17), 1).otherwise(0)) \
        .withColumn("is_peak_hours",
                   when((col("hour") >= 17) & (col("hour") <= 21), 1).otherwise(0))

    # Define window for lag features
    # Removed partitionBy("meter_id") as the column does not exist in the current schema
    window_spec = Window.orderBy("timestamp")

    # Lag features (previous consumption values)
    for lag_val in [1, 2, 3, 24, 48, 168]:  # 1-3h, 1-2 days, 1 week
        df = df.withColumn(
            f"energy_lag_{lag_val}",
            lag("Power_Consumption", lag_val).over(window_spec)
        )

    # Rolling window features
    # 24-hour window
    # Removed partitionBy("meter_id") as the column does not exist in the current schema
    window_24h = Window \
        .orderBy(col("timestamp").cast("long")) \
        .rangeBetween(-24*3600, 0)

    df = df \
        .withColumn("energy_mean_24h", avg("Power_Consumption").over(window_24h)) \
        .withColumn("energy_std_24h", stddev("Power_Consumption").over(window_24h)) \
        .withColumn("energy_min_24h", min("Power_Consumption").over(window_24h)) \
        .withColumn("energy_max_24h", max("Power_Consumption").over(window_24h))

    # 7-day window
    # Removed partitionBy("meter_id") as the column does not exist in the current schema
    window_7d = Window \
        .orderBy(col("timestamp").cast("long")) \
        .rangeBetween(-7*24*3600, 0)

    df = df \
        .withColumn("energy_mean_7d", avg("Power_Consumption").over(window_7d)) \
        .withColumn("energy_std_7d", stddev("Power_Consumption").over(window_7d))

    # Weather interaction features
    df = df \
        .withColumn("temp_hour_interaction", col("temperature") * col("hour")) \
        .withColumn("humidity_temp_interaction", col("humidity") * col("temperature"))

    # Remove rows with null lag features (beginning of time series)
    df = df.na.drop(subset=["energy_lag_1", "energy_lag_24"])

    # Rename 'Power_Consumption' to 'energy_kwh' for consistency with modeling steps
    df = df.withColumnRenamed("Power_Consumption", "energy_kwh")

    return df

# Apply feature engineering
featured_df = create_features(clean_df)

# Save processed data
featured_df.write \
    .mode("overwrite") \
    .parquet("processed_data/featured_data.parquet")

In [5]:
def create_train_test_split(df, train_ratio=0.6 ):
    """
    Split data chronologically for time series
    """
    # Get total count
    total_count = df.count()
    train_count = int(total_count * train_ratio)

    # Sort by timestamp
    df = df.orderBy("timestamp")

    # Add row number
    window = Window.orderBy("timestamp")
    df = df.withColumn("row_num", row_number().over(window))

    # Split
    train_df = df.filter(col("row_num") <= train_count).drop("row_num")
    test_df = df.filter(col("row_num") > train_count).drop("row_num")

    return train_df, test_df

# Create splits
train_df, test_df = create_train_test_split(featured_df)

print(f"Training samples: {train_df.count()}")
print(f"Test samples: {test_df.count()}")

Training samples: 29503
Test samples: 19669


In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from sklearn.preprocessing import StandardScaler
import pickle


def prepare_lstm_data(spark_df, sequence_length=24, scaler=None):
    """
    Convert Spark DataFrame to LSTM-ready numpy arrays.
    If scaler is None, it fits a new one (use for Training).
    If scaler is provided, it uses it (use for Test/Validation).
    """

    pdf = spark_df.toPandas()

    feature_cols = [
        'energy_kwh', 'temperature', 'humidity',
        'hour', 'day_of_week', 'is_weekend', 'is_peak_hours',
        'energy_lag_1', 'energy_lag_24', 'energy_mean_24h'
    ]

    if scaler is None:
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(pdf[feature_cols])
    else:
        scaled_data = scaler.transform(pdf[feature_cols])

    X_list, y_list = [], []

    # Vectorized check: ensure we have enough data
    if len(scaled_data) <= sequence_length:
        return np.array([]), np.array([]), scaler

    for i in range(len(scaled_data) - sequence_length):
        X_list.append(scaled_data[i : i + sequence_length])
        y_list.append(scaled_data[i + sequence_length, 0]) # Target: energy_kwh

    return np.array(X_list), np.array(y_list), scaler

# --- FIX 2: Modern Keras Input layer and Architecture ---
def build_advanced_lstm(sequence_length, n_features):
    model = Sequential([
        # Use Input layer instead of input_shape in the first layer for clarity
        Input(shape=(sequence_length, n_features)),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1) # Linear activation for regression
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='huber', # Good choice! Robust to outliers.
        metrics=['mae']
    )
    return model

# --- Execution ---
SEQUENCE_LENGTH = 24

print("Preparing LSTM training data...")
# Fit scaler on TRAIN only
X_train, y_train, train_scaler = prepare_lstm_data(train_df, SEQUENCE_LENGTH)

print("Preparing LSTM test data...")
# Apply TRAIN scaler to TEST
X_test, y_test, _ = prepare_lstm_data(test_df, SEQUENCE_LENGTH, scaler=train_scaler)



print(f"X_train shape: {X_train.shape}") # Expect: (Samples, 24, 12)

print("Building LSTM model...")
lstm_model = build_advanced_lstm(
    sequence_length=SEQUENCE_LENGTH,
    n_features=X_train.shape[2]
)

print("Training LSTM model...")
history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32, # Changed to power of 2 (common practice)
    verbose=1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5
        )
    ]
)

# Save results
lstm_model.save('models/lstm_energy_model.h5') # .keras is the modern standard
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(train_scaler, f)

Preparing LSTM training data...
Preparing LSTM test data...
X_train shape: (29479, 24, 10)
Building LSTM model...
Training LSTM model...
Epoch 1/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 162s 166ms/step - loss: 0.4400 - mae: 0.8183 - val_loss: 0.4363 - val_mae: 0.8143 - learning_rate: 0.0010
Epoch 2/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 135s 146ms/step - loss: 0.4343 - mae: 0.8118 - val_loss: 0.4362 - val_mae: 0.8143 - learning_rate: 0.0010
Epoch 3/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 140s 144ms/step - loss: 0.4357 - mae: 0.8134 - val_loss: 0.4375 - val_mae: 0.8156 - learning_rate: 0.0010
Epoch 4/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 142s 144ms/step - loss: 0.4318 - mae: 0.8070 - val_loss: 0.4362 - val_mae: 0.8143 - learning_rate: 0.0010
Epoch 5/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 134s 145ms/step - loss: 0.4354 - mae: 0.8130 - val_loss: 0.4363 - val_mae: 0.8144 - learning_rate: 0.0010
Epoch 6/50
922/922 ━━━━━━━━━━━━━━━━━━━━ 133s 145ms/step - loss: 0.4359 - mae: 0.8145 - val_loss: 0.4362 - val_mae: 0.8143 - learning_

In [7]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

def train_ensemble_models(train_df, test_df):
    """
    Train ensemble models using MLlib
    """
    # Define features
    feature_cols = [
        'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_hours',
        'temperature', 'humidity', 'temp_hour_interaction',
        'energy_lag_1', 'energy_lag_24', 'energy_lag_168',
        'energy_mean_24h', 'energy_std_24h', 'energy_mean_7d'
    ]

    # Assemble features
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features",
        handleInvalid="skip"
    )

    # Random Forest
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="energy_kwh",
        numTrees=200,
        maxDepth=15,
        minInstancesPerNode=5,
        seed=42
    )

    # Gradient Boosted Trees
    gbt = GBTRegressor(
        featuresCol="features",
        labelCol="energy_kwh",
        maxIter=100,
        maxDepth=10,
        stepSize=0.1,
        seed=42
    )

    # Train Random Forest
    print("Training Random Forest...")
    rf_pipeline = Pipeline(stages=[assembler, rf])
    rf_model = rf_pipeline.fit(train_df)

    # Train GBT
    print("Training Gradient Boosted Trees...")
    gbt_pipeline = Pipeline(stages=[assembler, gbt])
    gbt_model = gbt_pipeline.fit(train_df)

    # Evaluate models
    evaluator = RegressionEvaluator(
        labelCol="energy_kwh",
        predictionCol="prediction"
    )

    # RF predictions
    rf_predictions = rf_model.transform(test_df)
    rf_rmse = evaluator.setMetricName("rmse").evaluate(rf_predictions)
    rf_mae = evaluator.setMetricName("mae").evaluate(rf_predictions)
    rf_r2 = evaluator.setMetricName("r2").evaluate(rf_predictions)

    print(f"\nRandom Forest Metrics:")
    print(f"  RMSE: {rf_rmse:.4f}")
    print(f"  MAE: {rf_mae:.4f}")
    print(f"  R²: {rf_r2:.4f}")

    # GBT predictions
    gbt_predictions = gbt_model.transform(test_df)
    gbt_rmse = evaluator.setMetricName("rmse").evaluate(gbt_predictions)
    gbt_mae = evaluator.setMetricName("mae").evaluate(gbt_predictions)
    gbt_r2 = evaluator.setMetricName("r2").evaluate(gbt_predictions)

    print(f"\nGradient Boosted Trees Metrics:")
    print(f"  RMSE: {gbt_rmse:.4f}")
    print(f"  MAE: {gbt_mae:.4f}")
    print(f"  R²: {gbt_r2:.4f}")

    # Save models
    rf_model.write().overwrite().save("models/rf_energy_model")
    gbt_model.write().overwrite().save("models/gbt_energy_model")

    return rf_model, gbt_model

# Train ensemble models
rf_model, gbt_model = train_ensemble_models(train_df, test_df)

Training Random Forest...
Training Gradient Boosted Trees...

Random Forest Metrics:
  RMSE: 4.7237
  MAE: 3.8330
  R²: 0.0040

Gradient Boosted Trees Metrics:
  RMSE: 5.3948
  MAE: 4.3343
  R²: -0.2992


In [8]:
from pyspark.ml import PipelineModel
import tensorflow as tf

class HybridEnergyPredictor:
    def __init__(self, lstm_model_path, rf_model_path, gbt_model_path, scaler_path):
        # Load LSTM model
        self.lstm_model = tf.keras.models.load_model(lstm_model_path)

        # Load scaler
        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)

        # Load ensemble models
        self.rf_model = PipelineModel.load(rf_model_path)
        self.gbt_model = PipelineModel.load(gbt_model_path)

    def predict_lstm(self, sequence_data):
        """Make LSTM predictions"""
        return self.lstm_model.predict(sequence_data, verbose=0)

    def predict_ensemble(self, spark_df):
        """Make ensemble predictions"""
        rf_pred = self.rf_model.transform(spark_df)
        gbt_pred = self.gbt_model.transform(spark_df)

        return rf_pred, gbt_pred

    def hybrid_predict(self, spark_df, sequence_length=24):
        """
        Combine LSTM and ensemble predictions
        """
        # Prepare LSTM data
        X_lstm, _, _ = prepare_lstm_data(spark_df, sequence_length)
        lstm_predictions = self.predict_lstm(X_lstm)

        # Get ensemble predictions
        rf_pred, gbt_pred = self.predict_ensemble(spark_df)

        # Extract predictions to arrays
        rf_values = np.array([row['prediction'] for row in
                             rf_pred.select('prediction').collect()])
        gbt_values = np.array([row['prediction'] for row in
                              gbt_pred.select('prediction').collect()])

        # Align array sizes (LSTM has fewer predictions due to sequence window)
        min_len = __builtins__.min(len(lstm_predictions), len(rf_values), len(gbt_values))
        lstm_predictions = lstm_predictions[-min_len:]
        rf_values = rf_values[-min_len:]
        gbt_values = gbt_values[-min_len:]

        # Weighted ensemble
        weights = {
            'lstm': 0.5,
            'rf': 0.25,
            'gbt': 0.25
        }

        final_predictions = (
            weights['lstm'] * lstm_predictions.flatten() +
            weights['rf'] * rf_values +
            weights['gbt'] * gbt_values
        )

        return final_predictions

# Initialize predictor
predictor = HybridEnergyPredictor(
    lstm_model_path='models/lstm_energy_model.h5',
    rf_model_path='models/rf_energy_model',
    gbt_model_path='models/gbt_energy_model',
    scaler_path='models/scaler.pkl'
)

In [9]:
import os

def batch_predict_and_save(input_csv_path, output_csv_path):
    """
    Read CSV, make predictions, save results
    """
    # Read new data
    new_data = spark.read \
        .option("header", "true") \
        .schema(schema) \
        .csv(input_csv_path)

    # Preprocess
    new_data = preprocess_csv_data(new_data)
    new_data = create_features(new_data)

    # Make predictions
    predictions = predictor.hybrid_predict(new_data)

    # Add predictions to DataFrame
    pdf = new_data.toPandas()
    pdf['predicted_energy_kwh'] = np.nan
    pdf.iloc[-len(predictions):, pdf.columns.get_loc('predicted_energy_kwh')] = predictions

    # Calculate prediction interval (simple approach)
    pdf['prediction_lower'] = pdf['predicted_energy_kwh'] * 0.9
    pdf['prediction_upper'] = pdf['predicted_energy_kwh'] * 1.1

    # Ensure the output directory exists
    output_dir = os.path.dirname(output_csv_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Save to CSV
    pdf.to_csv(output_csv_path, index=False)
    print(f"Predictions saved to {output_csv_path}")

    # Also save as Parquet for faster access
    spark.createDataFrame(pdf).write \
        .mode("overwrite") \
        .parquet(output_csv_path.replace('.csv', '.parquet'))

    return pdf

# Run batch prediction
results = batch_predict_and_save(
    input_csv_path="/content/smart_grid_dataset.csv",
    output_csv_path="predictions/energy_forecast.csv"
)

Predictions saved to predictions/energy_forecast.csv


In [ ]:
def main_pipeline():
    """
    Complete energy forecasting pipeline for CSV data
    """
    print("=" * 60)
    print("Energy Consumption Forecasting Pipeline")
    print("=" * 60)

    # 1. Load data
    print("\n[Step 1] Loading CSV data...")
    df = spark.read \
        .option("header", "true") \
        .schema(schema) \
        .csv("/content/smart_grid_dataset.csv")

    print(f"Loaded {df.count()} records")

    # 2. Preprocess
    print("\n[Step 2] Preprocessing data...")
    clean_df = preprocess_csv_data(df)
    print(f"After cleaning: {clean_df.count()} records")

    # 3. Feature engineering
    print("\n[Step 3] Creating features...")
    featured_df = create_features(clean_df)

    # Cache for performance
    featured_df.cache()

    # 4. Train/test split
    print("\n[Step 4] Creating train/test split...")
    train_df, test_df = create_train_test_split(featured_df, train_ratio=0.8)

    # 5. Train LSTM
    print("\n[Step 5] Training LSTM model...")
    X_train, y_train, scaler = prepare_lstm_data(train_df)
    X_test, y_test, _ = prepare_lstm_data(test_df)

    lstm_model = build_advanced_lstm(24, X_train.shape[2])
    lstm_model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=30,
        batch_size=64,
        verbose=1
    )

    lstm_model.save('models/lstm_energy_model.h5')

    # 6. Train ensemble models
    print("\n[Step 6] Training ensemble models...")
    rf_model, gbt_model = train_ensemble_models(train_df, test_df)

    # 7. Make predictions on test set
    print("\n[Step 7] Making predictions...")
    predictor = HybridEnergyPredictor(
        lstm_model_path='models/lstm_energy_model.h5',
        rf_model_path='models/rf_energy_model',
        gbt_model_path='models/gbt_energy_model',
        scaler_path='models/scaler.pkl'
    )

    predictions = predictor.hybrid_predict(test_df)

    # 8. Evaluate
    print("\n[Step 8] Evaluating hybrid model...")
    test_pdf = test_df.toPandas()
    actual = test_pdf['energy_kwh'].values[-len(predictions):]

    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    rmse = np.sqrt(mean_squared_error(actual, predictions))
    mae = mean_absolute_error(actual, predictions)
    r2 = r2_score(actual, predictions)
    mape = np.mean(np.abs((actual - predictions) / actual)) * 100

    print(f"\nHybrid Model Performance:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R²: {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")

    # 9. Save predictions
    print("\n[Step 9] Saving predictions...")
    test_pdf['predicted_energy_kwh'] = np.nan
    test_pdf.iloc[-len(predictions):,
                  test_pdf.columns.get_loc('predicted_energy_kwh')] = predictions

    test_pdf.to_csv('predictions/test_predictions.csv', index=False)

    print("\n" + "=" * 60)
    print("Pipeline completed successfully!")
    print("=" * 60)

    return predictor, test_pdf

if __name__ == "__main__":
    predictor, results = main_pipeline()

Energy Consumption Forecasting Pipeline

[Step 1] Loading CSV data...
Loaded 50000 records

[Step 2] Preprocessing data...
After cleaning: 49196 records

[Step 3] Creating features...

[Step 4] Creating train/test split...

[Step 5] Training LSTM model...
Epoch 1/30
615/615 ━━━━━━━━━━━━━━━━━━━━ 134s 202ms/step - loss: 0.4378 - mae: 0.8161 - val_loss: 0.4333 - val_mae: 0.8102
Epoch 2/30
615/615 ━━━━━━━━━━━━━━━━━━━━ 119s 194ms/step - loss: 0.4337 - mae: 0.8115 - val_loss: 0.4330 - val_mae: 0.8097
Epoch 3/30
615/615 ━━━━━━━━━━━━━━━━━━━━ 120s 196ms/step - loss: 0.4345 - mae: 0.8121 - val_loss: 0.4338 - val_mae: 0.8106
Epoch 4/30
615/615 ━━━━━━━━━━━━━━━━━━━━ 123s 200ms/step - loss: 0.4360 - mae: 0.8135 - val_loss: 0.4328 - val_mae: 0.8095
Epoch 5/30
615/615 ━━━━━━━━━━━━━━━━━━━━ 117s 189ms/step - loss: 0.4356 - mae: 0.8135 - val_loss: 0.4328 - val_mae: 0.8096
Epoch 6/30
615/615 ━━━━━━━━━━━━━━━━━━━━ 142s 190ms/step - loss: 0.4296 - mae: 0.8062 - val_loss: 0.4328 - val_mae: 0.8095
Epoch 7/30
6

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os # Import the os module for directory operations

def visualize_results(results_df, meter_id=None, save_path='plots/'):
    """
    Create visualizations of predictions
    """
    # Ensure the save_path directory exists
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    if meter_id:
        plot_df = results_df[results_df['meter_id'] == meter_id].copy()
    else:
        plot_df = results_df.copy()

    plot_df = plot_df.sort_values('timestamp')

    # Time series plot
    plt.figure(figsize=(15, 6))
    plt.plot(plot_df['timestamp'], plot_df['energy_kwh'],
             label='Actual', alpha=0.7, linewidth=2)
    plt.plot(plot_df['timestamp'], plot_df['predicted_energy_kwh'],
             label='Predicted', alpha=0.7, linewidth=2)
    plt.xlabel('Timestamp')
    plt.ylabel('Energy Consumption (kWh)')
    plt.title('Energy Consumption: Actual vs Predicted')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{save_path}time_series.png', dpi=300)
    plt.close()

    # Scatter plot
    plt.figure(figsize=(8, 8))
    plt.scatter(plot_df['energy_kwh'], plot_df['predicted_energy_kwh'], alpha=0.5)
    plt.plot([plot_df['energy_kwh'].min(), plot_df['energy_kwh'].max()],
             [plot_df['energy_kwh'].min(), plot_df['energy_kwh'].max()],
             'r--', linewidth=2)
    plt.xlabel('Actual Energy (kWh)')
    plt.ylabel('Predicted Energy (kWh)')
    plt.title('Prediction Accuracy')
    plt.tight_layout()
    plt.savefig(f'{save_path}scatter_plot.png', dpi=300)
    plt.close()

    print(f"Visualizations saved to {save_path}")

# Create visualizations
visualize_results(results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

# Define the base directory in Google Drive where outputs will be saved
drive_output_base_path = '/content/drive/MyDrive/energy_forecasting_outputs'

# Create the base directory if it doesn't exist
os.makedirs(drive_output_base_path, exist_ok=True)
print(f"Google Drive output directory created at: {drive_output_base_path}")

# Define subdirectories for models, predictions, plots, and processed data
os.makedirs(os.path.join(drive_output_base_path, 'models'), exist_ok=True)
os.makedirs(os.path.join(drive_output_base_path, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(drive_output_base_path, 'plots'), exist_ok=True)
os.makedirs(os.path.join(drive_output_base_path, 'processed_data'), exist_ok=True)

print("Subdirectories 'models', 'predictions', 'plots', and 'processed_data' created within the output directory.")

In [ ]:
import shutil
import os

# Define the base paths
local_base_path = '/content'
drive_output_base_path = '/content/drive/MyDrive/energy_forecasting_outputs'

# List of files and directories to copy
outputs_to_copy = [
    'processed_data/featured_data.parquet',
    'models/lstm_energy_model.h5',
    'models/scaler.pkl',
    'models/rf_energy_model', # This is a directory
    'models/gbt_energy_model', # This is a directory
    'predictions/energy_forecast.csv',
    'predictions/energy_forecast.parquet',
    'predictions/test_predictions.csv',
    'plots/time_series.png',
    'plots/scatter_plot.png'
]

print("Starting to copy output files to Google Drive...")

for output_item in outputs_to_copy:
    local_path = os.path.join(local_base_path, output_item)
    drive_path = os.path.join(drive_output_base_path, output_item)

    # Ensure parent directory exists in Drive (should be already created)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)

    if os.path.isdir(local_path):
        # For directories, use copytree. Remove existing destination if it's a directory
        if os.path.exists(drive_path):
            shutil.rmtree(drive_path)
        shutil.copytree(local_path, drive_path)
        print(f"Copied directory: {local_path} to {drive_path}")
    elif os.path.isfile(local_path):
        # For files, use copy
        shutil.copy(local_path, drive_path)
        print(f"Copied file: {local_path} to {drive_path}")
    else:
        print(f"Warning: {local_path} does not exist or is not a file/directory.")

print("All output files have been copied to Google Drive.")